In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import shuffle
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import seaborn as sns   
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler  
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical                                  
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input  
from tensorflow.keras.models import Sequential                                     
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.layers import BatchNormalization                             
from tensorflow.keras.optimizers import Adam                                       

plt.rcParams['font.sans-serif'] = ['SimHei']  # 黑体
plt.rcParams['axes.unicode_minus'] = False   

In [ ]:
# 固定随机种子
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
#读取
data = pd.read_excel(
    r'数据集路径'
) 
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [ ]:
# 前面均分，最后一列是标签
X_raw = data.iloc[:, :-1].values
y_raw = data.iloc[:, -1].values

timesteps = 100
channels = 3
# 检查列数是否为 timesteps * channels
#assert X_raw.shape[1] == timesteps * channels, "特征列数与 timesteps*channels 不匹配"

# 重塑为 (n_samples, timesteps, channels)
X = X_raw.reshape(X_raw.shape[0], channels, timesteps).transpose(0, 2, 1)
# 现在 X.shape -> (n_samples, 100, 3)


In [ ]:
# 标准化
for ch in range(channels):
    sc = StandardScaler()
    X[:, :, ch] = sc.fit_transform(X[:, :, ch])

In [ ]:
# 标签编码
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
y_cat = to_categorical(y_enc)

In [ ]:
# 划分训练集、测试集和验证集
# 验证形状
# 三路划分
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.3, random_state=SEED, stratify=y_enc
)
y_temp_labels = np.argmax(y_temp, axis=1)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED,
    stratify=y_temp_labels
)

In [ ]:
#1D-CNN
model = Sequential([
    Input(shape=(timesteps, channels)),

    Conv1D(128, kernel_size=3, activation='relu',
           kernel_regularizer=l2(1e-3)),
    MaxPooling1D(2),
    Dropout(0.3),

    Conv1D(64, kernel_size=3, activation='relu',
           kernel_regularizer=l2(1e-3)),
    MaxPooling1D(2),
    Dropout(0.3),

    Flatten(),
    Dense(32, activation='relu', kernel_regularizer=l2(1e-3)),
    Dropout(0.3),
    Dense(y_cat.shape[1], activation='softmax')
])

In [ ]:

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss', patience=30,
    restore_best_weights=True
)

In [ ]:
# 训练模型
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=300,
    batch_size=16,
    callbacks=[early_stop],
    verbose=2
)

In [ ]:
# 评估
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n测试集准确率: {acc*100:.2f}%")

y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)
print(classification_report(
    y_true_labels, y_pred_labels, target_names=le.classes_
))

In [ ]:
history_df = pd.DataFrame(history.history)

history_df.insert(0, 'epoch', range(1, len(history_df) + 1))

history_df.rename(columns={
    'loss':         'Train_Loss',
    'val_loss':     'Val_Loss',
    'accuracy':     'Train_Accuracy',
    'val_accuracy': 'Val_Accuracy',
   
}, inplace=True)

# 保存为 Excel
output_path = r'输出地址'
history_df.to_excel(output_path, index=False, sheet_name='Training History')

print(f"训练过程数据已保存至：{output_path}")
print(history_df.head())
